# Toda la API: de Tool a Eval · 2.1.4

Input subtotales 100 y 200 → Context, Tool, Skill, Agent, decisión del System y Environment → total acumulado 348. La fuente es visible aquí; Python, sin inferencia.

In [2]:
"""A small, offline invoice: Tool → Agent → System → Environment, with Eval."""
import pydantic
import agentic_systems as toolkit

class Invoice(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    status: str | None = None

@toolkit.tool
def quote(subtotal: int, *, context: toolkit.Context) -> dict:
    """Calculate integer money using the domain's tax policy."""
    return {'total': subtotal * (100 + context.data['tax_percent']) // 100, 'currency': context.data['currency']}
domain = toolkit.Context({'tax_percent': 16, 'currency': 'MXN'}, sources=('tax-policy-v1',))
capability = toolkit.skill(name='invoicing', tools=(quote,), instructions='Amounts are integer MXN.')
memory = toolkit.Memory(capacity=3)
agent_graph = toolkit.Graph(Invoice)
agent_graph.add_node('quote', quote)
agent_graph.add_edge(toolkit.START, 'quote')
agent_graph.add_edge('quote', toolkit.END)
agent = toolkit.Agent('accountant', graph=agent_graph, skills=(capability,), context=domain, memory=memory)
system_graph = toolkit.Graph(Invoice)
system_graph.add_node('accountant', agent)
def approve_step(state):
    return {'status': 'approved'}
system_graph.add_node('approve', approve_step)
def review_step(state):
    return {'status': 'review'}
system_graph.add_node('review', review_step)
system_graph.add_edge(toolkit.START, 'accountant')
system_graph.add_conditional_edges('accountant', lambda state: 'review' if state.total > 1000 else 'approve', ['approve', 'review'])
system_graph.add_edge('approve', toolkit.END)
system_graph.add_edge('review', toolkit.END)
system = toolkit.System('billing', graph=system_graph, context=domain)
environment_graph = toolkit.Graph(Invoice)
environment_graph.add_node('billing', system)
environment_graph.add_edge(toolkit.START, 'billing')
environment_graph.add_edge('billing', toolkit.END)
environment = toolkit.Environment('batch', graph=environment_graph, records=({'subtotal': 100}, {'subtotal': 200}), state={'sum': 0}, update=lambda state, record, result: {'sum': state['sum'] + result.data['total']})
evaluation = toolkit.Eval('invoice-contract', system, cases=({'input': {'subtotal': 100}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['total'] == 116 and result.data['currency'] == 'MXN'},))


In [3]:
single = agent.run({'subtotal': 100})
team = system.run({'subtotal': 100})
batch = environment.run()
checked = evaluation.run()
assert single.ok and team.ok and batch.ok and checked.ok
assert batch.data['summary']['state'] == {'sum': 348}
print(single.data, team.data, batch.data['summary'])
print(batch.lineage().render())

print(single.human_results())


{'subtotal': 100, 'total': 116, 'currency': 'MXN'} {'subtotal': 100, 'total': 116, 'currency': 'MXN', 'status': 'approved'} {'state': {'sum': 348}, 'processed': 2, 'terminated': True, 'truncated': False, 'reason': 'records_exhausted'}
run: OK
{"preview": "{\"outputs\":[{\"subtotal\":100,\"total\":116,\"currency\":\"MXN\",\"status\":\"approved\"},{\"subtotal\":200,\"total\":232,\"currency\":\"MXN\",\"status\":\"approved\"}],\"history\":[{\"step\":0,\"record\":{\"subtotal\":100},\"state_before\":{\"sum\":0},\"state_after\":{\"sum\":116},\"execution_id\":\"run-ca8269dc93cb47e59a9ba1d495ddb5db\"},{\"step\":1,\"record\":{\"subtotal\":200},\"state_before\":{\"sum\":116},\"state_after\":{\"sum\":348},\"execution_id", "omitted": true}
- batch: completed [run-16ddc037dd0c4a89a6c7895fa961bb45]
- observe: completed [run-10566cd81f9c452ca41446cd24590d7c]
- Next node: run [run-10566cd81f9c452ca41446cd24590d7c:decision]
- Next node: approve [run-67239f5ef12a4833b143944b282abced:decision]
- quote: co

Carga local de una Skill; el código Python de la carpeta debe ser confiable.

In [4]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/common_graph/skills/pricing').is_dir()))
loaded = toolkit.load_skill(ROOT / 'examples/common_graph/skills/pricing')
assert loaded.tools
print(loaded.inspect())


{'name': 'pricing', 'description': '', 'version': '1.0.0', 'instructions': 'Use the currency and tax policy supplied in Context.', 'tools': [{'name': 'quote', 'description': 'Apply an integer percentage from the explicit policy snapshot.', 'metadata': {}, 'input_schema': None, 'output_schema': None}], 'metadata': {}}


Una función nodo proyecta explícitamente la entrada y llama a la Tool. La evidencia se conserva automáticamente.

In [5]:
class Nested(pydantic.BaseModel):
    order: Invoice
    total: int | None = None
    currency: str | None = None
projection = toolkit.Graph(Nested)
def quote_step(state):
    return quote.run(state.order.model_dump(mode="json", exclude_unset=True))
projection.add_node('quote', quote_step)
projection.add_edge(toolkit.START, 'quote')
projection.add_edge('quote', toolkit.END)
projected = toolkit.Agent('nested', graph=projection, context=domain).run({'order': {'subtotal': 100}})
assert projected.ok and projected.data['total'] == 116
print(projected.data)


{'order': {'subtotal': 100}, 'total': 116, 'currency': 'MXN'}


runtime y framework seleccionan un actor atómico; LangGraph sigue coordinando.

In [6]:
operator = toolkit.Agent('operator', runtime=toolkit.runtime(provider='python-runtime'), framework=toolkit.framework('native'), skills=(capability,), context=domain)
operated = operator.run({'subtotal': 100}, timeout_s=10)
assert operated.ok and operated.data['total'] == 116
restored = toolkit.RunResult(**operated.to_dict())
assert restored.check_invariants().ok
print(restored.data)


{'total': 116, 'currency': 'MXN'}


Pausa y reanudación dentro del grafo, en el mismo proceso y programa.

In [7]:
approval = toolkit.Graph(Invoice)
def approval_step(state):
    return {'status': 'approved' if toolkit.pause({'question': 'Approve?'}) else 'rejected'}
approval.add_node('approval', approval_step)
approval.add_edge(toolkit.START, 'approval')
approval.add_edge('approval', toolkit.END)
program = toolkit.System('approval', graph=approval).compile()
waiting = program.run({'subtotal': 100})
assert waiting.status == 'paused'
resumed = program.resume(waiting, True)
assert resumed.ok and resumed.data['status'] == 'approved'
print(resumed.data)


{'subtotal': 100, 'status': 'approved'}


Recursos de serving: declarar no inicia procesos, no descarga pesos ni valida GPU.

In [8]:
artifact = toolkit.model_artifact('Qwen/Qwen3-0.6B', revision='main', metadata={'scope': 'declaration only; resolved weights pending'})
server = toolkit.model_server(artifact, max_model_len=2048, max_num_seqs=1, gpu_memory_utilization=0.75)
print(artifact.inspect())
print(server.inspect())


{'model': 'Qwen/Qwen3-0.6B', 'base_model': None, 'adapter_path': None, 'tokenizer': None, 'revision': 'main', 'quantization': None, 'metadata': {'scope': 'declaration only; resolved weights pending'}}


{'backend': 'vllm', 'installed': False, 'command': ['vllm', 'serve', 'Qwen/Qwen3-0.6B', '--host', '127.0.0.1', '--port', '8000', '--served-model-name', 'Qwen/Qwen3-0.6B', '--gpu-memory-utilization', '0.75', '--max-model-len', '2048', '--max-num-seqs', '1', '--enable-auto-tool-choice', '--tool-call-parser', 'hermes', '--generation-config', 'vllm', '--api-key', '[REDACTED]'], 'endpoint': {'url': 'http://127.0.0.1:8000/v1', 'model': 'Qwen/Qwen3-0.6B', 'backend': 'vllm', 'api_key_configured': '[REDACTED]', 'pid': None, 'owned': False}, 'configuration': {'schema_version': 'agentic_systems.serving.v1', 'backend': 'vllm', 'artifact': {'schema_version': 'agentic_systems.serving.v1', 'adapter_path': None, 'revision': 'main', 'quantization': None, 'metadata': {'scope': 'declaration only; resolved weights pending'}, 'model': 'Qwen/Qwen3-0.6B', 'base_model': None, 'tokenizer': None}, 'profile': 'fast', 'served_model_name': None, 'host': '127.0.0.1', 'port': 8000, 'gpu_memory_utilization': 0.75, 'm